# 15 · The Avengers Initiative
*The gold layer finally has customers.*

Until now the gold tables only sat there. This chapter gives them two consumers: a **dashboard** for the field teams (AI/BI dashboards on a SQL warehouse)
and a **model** that predicts serious incidents (scikit-learn, tracked and registered with MLflow). The data is random, so the model will be useless.
That is the point: the plumbing is what you are learning, and a pipeline that reports "AUC ≈ 0.5" honestly is a good pipeline.

| Concept | Where you meet it |
|---|---|
| The serving layer: who reads gold, and through what | the next section |
| A SQL warehouse and dashboard datasets (AI/BI dashboards, GA) | Steps 1 and 2 |
| Features and a label from silver (one-hot encoding) | Step 3 |
| Train/test split and AUC | Steps 4 and 5 |
| MLflow tracking and a model in the Unity Catalog registry | the "Models in Unity Catalog" section, Step 5 |

Deep dive: [docs/15-avengers-initiative.md](../../docs/15-avengers-initiative.md)

**Needs:**
* Chapter 06 ran (`shield.silver.incident_reports` with 9,500 validated incidents, `shield.silver.locations`, and the gold tables `shield.gold.daily_threat_briefing` and `shield.gold.location_risk` rebuilt from it). If you only ran chapter 05, silver still has the 500 defective incidents and the **Expected** numbers below will show ❌. Chapter 03 alone is not enough: the notebook stops in Setup when a table is missing.
* Compute that has `scikit-learn`, `pandas` and `mlflow` (MLflow 3 for `name=` in `log_model`; older MLflow uses `artifact_path=`). Databricks Runtime ML has them; whether serverless notebook compute has them is not verified here.
* Permission to create a model in `shield.gold` (`CREATE MODEL` and `USE SCHEMA`) and, for Steps 1 and 2, `CAN USE` on a SQL warehouse plus the Databricks SQL entitlement.
* The **Expected** numbers assume the dataset from chapter 02 (10,000 incidents, seed 42, 5 % dirty); other data gives other numbers and a ❌, which is not a bug.

**Costs money and leaves state:** a running SQL warehouse, an MLflow experiment, and a registered model in Unity Catalog. The 🧹 Clean up section removes the model; delete the dashboard and stop the warehouse yourself.

## The serving layer

**The idea:** a gold table is only worth what people and systems get out of it. The *serving layer* is every way gold is read: a dashboard, a BI tool, an API, a model. It is a role, not a Databricks feature, and the data engineer's job there is a *stable, documented gold table*, not the chart.

```
 shield.silver ──▶ shield.gold ──┬──▶ SQL warehouse ──▶ AI/BI dashboard        (Steps 1-2)
                                 │
                                 └──▶ features ──▶ model ──▶ MLflow run ──▶ Unity Catalog model   (Steps 3-5)
```

| Consumer | Reads | Through | Built in |
|---|---|---|---|
| Field team dashboard | `gold.daily_threat_briefing`, `gold.location_risk` | a SQL warehouse (compute tuned for SQL and BI) | Steps 1 and 2 |
| Threat-ranking model | `silver.incident_reports` | a notebook, then MLflow | Steps 3 to 5 |

**How this notebook implements it**

* **Dashboard:** three SQL queries become the *datasets* of an AI/BI dashboard. Step 1 runs them here first, so you know they return what the charts need. Step 2 is the click-through in the UI (there is no code for it).
* **Model:** a random forest classifier predicts `serious` (severity `high` or `critical`) from location, incident type, hour and weekday. The label comes from silver, not gold, because gold has already lost the individual rows.
* **Status today** (checked against Microsoft Learn, not by running it): AI/BI dashboards are generally available and the older "legacy" dashboards have reached end of life. Genie, the natural-language question box on gold tables, is generally available too, but your admin has to allow partner-powered AI features, and the docs now call a Genie space a "Genie Agent". Whether Model Serving is GA or preview for your account is something to check in the docs before promising it.

**Why it helps:** the same gold tables feed every consumer, so a fix in silver reaches the dashboard and the next training run. Dashboards query live tables, so a gold table that changes shape breaks every chart on it.

**Not covered yet:** scheduled dashboard refresh and scoring jobs (see the doc's mini-projects), a serving endpoint, and Genie. Chapter 16 only checks that gold is healthy.

## Setup

In [ ]:
import mlflow
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

MODEL_NAME = "shield.gold.serious_incident_model"
mlflow.set_registry_uri("databricks-uc")  # register models in Unity Catalog, not the legacy workspace registry


def show_checks(checks):
    """Show (check, result, expected) rows with a ✅ / ❌ for whether result matches expected."""
    fmt = lambda v: f"{v:,}" if isinstance(v, int) and not isinstance(v, bool) else str(v)
    rows = [(what, fmt(got), fmt(want), "✅" if got == want else "❌") for what, got, want in checks]
    display(spark.createDataFrame(rows, "check string, result string, expected string, ok string"))


needed = [
    "shield.silver.incident_reports",
    "shield.silver.locations",
    "shield.gold.daily_threat_briefing",
    "shield.gold.location_risk",
]
missing = [t for t in needed if not spark.catalog.tableExists(t)]
assert not missing, f"Run notebooks 03 and 06 first. Missing tables: {missing}"
assert spark.table("shield.silver.incident_reports").count() > 0, "silver.incident_reports is empty: run notebook 06 first"

## 1 · The dashboard

### Step 1 · Check the dataset queries
**Goal:** prove that the three queries a dashboard would use return what the charts need, before you paste them into the UI.

| Dataset | Query reads | Chart it feeds |
|---|---|---|
| `incidents_by_day` | `gold.daily_threat_briefing` | incidents per day (line) |
| `risk_by_city` | `gold.location_risk` | serious incidents per city (bar) |
| `severity_mix` | `gold.daily_threat_briefing` | share per severity (pie) |

**Expected:** with the chapter-06 data there is one row per day for 2025 (365), the incidents add up to 9,500, there are 10 cities with Budapest on top (214 serious incidents), and 4 severities. The check table shows:

| check | result | expected | ok |
|---|---|---|---|
| incidents_by_day: days | 365 | 365 | ✅ |
| incidents_by_day: incidents in total | 9,500 | 9,500 | ✅ |
| risk_by_city: cities | 10 | 10 | ✅ |
| risk_by_city: most serious incidents | Budapest (214) | Budapest (214) | ✅ |
| severity_mix: severities | 4 | 4 | ✅ |

The queries select only the columns the charts need on purpose: viewers of a dashboard can reach everything in a dataset, even columns no chart shows. The cell only reads, so it is safe to re-run.

In [ ]:
QUERIES = {
    "incidents_by_day": "SELECT report_date, SUM(incidents) AS incidents FROM shield.gold.daily_threat_briefing GROUP BY 1 ORDER BY 1",
    "risk_by_city": "SELECT city, serious_incidents, incidents FROM shield.gold.location_risk ORDER BY serious_incidents DESC",
    "severity_mix": "SELECT severity, SUM(incidents) AS incidents FROM shield.gold.daily_threat_briefing GROUP BY 1",
}
for name, query in QUERIES.items():
    print(f"-- {name}\n{query}\n")

dataset_rows = {name: spark.sql(query).collect() for name, query in QUERIES.items()}
top_city = dataset_rows["risk_by_city"][0]
show_checks([
    ("incidents_by_day: days", len(dataset_rows["incidents_by_day"]), 365),
    ("incidents_by_day: incidents in total", sum(row["incidents"] for row in dataset_rows["incidents_by_day"]), 9_500),
    ("risk_by_city: cities", len(dataset_rows["risk_by_city"]), 10),
    ("risk_by_city: most serious incidents", f"{top_city['city']} ({top_city['serious_incidents']})", "Budapest (214)"),
    ("severity_mix: severities", len(dataset_rows["severity_mix"]), 4),
])

### Step 2 · Build the dashboard (in the UI, no code)
**Goal:** turn the three queries into a dashboard that someone else can open.

**Input:** the three queries printed in Step 1, and a SQL warehouse (serverless starts in seconds and stops when idle; create one under **SQL Warehouses** if you have none).

1. **New → Dashboard**. Open the **Data** tab and choose **Add SQL dataset** (older UI text: *Create from SQL*) once per query. Paste the query, press **Run**, rename the dataset.
2. On the canvas, add a line chart on `incidents_by_day`, a bar chart on `risk_by_city` and a pie chart on `severity_mix`.
3. **Publish** it, and set the warehouse it should use.

**Expected** (not verified in a workspace; this step is the UI): each dataset shows the same rows as Step 1, and the published dashboard renders three charts. A *draft* dashboard can only be shared with people inside the workspace.

**Cost:** the warehouse bills while it runs, including while a dashboard refreshes. Stop it when you are done (see 🧹 Clean up).

## 2 · The model

### Step 3 · Features and label from silver
**Goal:** turn incident rows into a table a model can learn from: numeric **features** (the inputs) and one **label** (what we predict).

| Column | Role | How it becomes numeric |
|---|---|---|
| `location_id` (10 values) | feature | *one-hot encoding*: one 0/1 column per value |
| `incident_type` (6 values) | feature | one-hot encoding |
| `hour`, `dow` (day of week) | features | already numbers |
| `serious` | label | 1 when `severity` is `high` or `critical`, otherwise 0 |

Only things known when the report arrives are features; `severity` is the label, so it must not appear among the features.

**Expected:** 9,500 rows and 18 feature columns (2 + 10 + 6). 1,863 incidents are serious, about 20 %. Silver is only read, so the cell is safe to re-run. `toPandas()` pulls the whole table to the driver; that is fine for 9,500 rows and would not be for 500 million.

| check | result | expected | ok |
|---|---|---|---|
| rows | 9,500 | 9,500 | ✅ |
| feature columns | 18 | 18 | ✅ |
| serious incidents | 1,863 | 1,863 | ✅ |

In [ ]:
incident_features = spark.sql(
    """
    SELECT location_id, incident_type, hour(reported_ts) AS hour, dayofweek(reported_ts) AS dow,
           CASE WHEN severity IN ('high','critical') THEN 1 ELSE 0 END AS serious
    FROM shield.silver.incident_reports
    """
).toPandas()

features = pd.get_dummies(incident_features.drop(columns="serious"), columns=["location_id", "incident_type"])
serious = incident_features["serious"]

show_checks([
    ("rows", len(features), 9_500),
    ("feature columns", features.shape[1], 18),
    ("serious incidents", int(serious.sum()), 1_863),
])

### Step 4 · Hold out a test set
**Goal:** keep rows the model never sees while training, so the score measures prediction and not memory.

**Expected:** a fixed 80 / 20 split (`random_state=42`): 7,600 training rows and 1,900 test rows. The split is by position in the table, and Spark does not promise a row order, so the *rows* in each part can differ between runs even though the sizes do not.

| check | result | expected | ok |
|---|---|---|---|
| training rows | 7,600 | 7,600 | ✅ |
| test rows | 1,900 | 1,900 | ✅ |

In [ ]:
features_train, features_test, serious_train, serious_test = train_test_split(features, serious, test_size=0.2, random_state=42)

show_checks([
    ("training rows", len(features_train), 7_600),
    ("test rows", len(features_test), 1_900),
])

## Models in Unity Catalog

**The idea:** *MLflow tracking* records what a training run did (parameters, metrics, the model file). The *Unity Catalog model registry* then keeps the models you decide to keep, as governed objects with versions, like tables. Both are features of Databricks and MLflow; which models are worth registering is your team's rule.

```
 experiment (the notebook)
   └── run "serious_incident_rf"      metric: auc
         └── logged model "model"  ──register──▶  shield.gold.serious_incident_model
                                                      ├── version 1
                                                      └── version 2 ...   (one more per run)
```

**How this notebook implements it**

* `mlflow.set_registry_uri("databricks-uc")` (Setup) sends registrations to Unity Catalog.
* Step 5 starts one run, trains the forest, logs the metric `auc`, and calls `mlflow.sklearn.log_model(..., registered_model_name=MODEL_NAME)`. The name must have three levels, `catalog.schema.model`, so the model lives in `shield.gold` and follows that schema's grants (chapter 11).
* `input_example` lets MLflow infer the model *signature* (the input columns and types). Unity Catalog refuses model versions without a signature.
* In a Databricks notebook, runs are recorded in the notebook's own experiment, so there is no experiment to create.

**Why it helps:** you can reproduce and compare runs, and later scoring jobs or endpoints refer to `shield.gold.serious_incident_model` and a version, not to a pickle file somebody emailed.

**Not covered yet:** model aliases (such as `champion`), loading a version for batch scoring, and serving endpoints. The doc's mini-projects point at them.

## 3 · Track and register

### Step 5 · Train, log and register
**Goal:** one MLflow run with an `auc` metric, and one new version of the model in Unity Catalog.

*AUC* is the chance that a randomly chosen serious incident gets a higher score than a randomly chosen ordinary one: 1.0 is perfect, **0.5 is a coin flip**. The generator draws every column independently, so there is nothing to learn.

**Expected:** AUC close to 0.5. A stand-in run on the same data gave values between 0.49 and 0.51; this notebook's exact value is not known until you run it, so the check accepts 0.45 to 0.55. The model gets a version in `shield.gold.serious_incident_model`.

**Not safe to re-run without a trace:** every run adds a run to the experiment and a **new model version**. The check only asks for at least one version.

| check | result | expected | ok |
|---|---|---|---|
| AUC between 0.45 and 0.55 | True | True | ✅ |
| model has at least one version | True | True | ✅ |

The printed line also shows the AUC itself, for example `AUC = 0.500`.

In [ ]:
with mlflow.start_run(run_name="serious_incident_rf"):
    model = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42).fit(features_train, serious_train)
    auc = roc_auc_score(serious_test, model.predict_proba(features_test)[:, 1])
    mlflow.log_metric("auc", auc)
    mlflow.sklearn.log_model(
        model, name="model", input_example=features_train.head(5), registered_model_name=MODEL_NAME
    )
print(f"AUC = {auc:.3f}  (about 0.5 = no signal, as expected for synthetic random data)")

model_versions = mlflow.MlflowClient().search_model_versions(f"name = '{MODEL_NAME}'")
show_checks([
    ("AUC between 0.45 and 0.55", 0.45 <= auc <= 0.55, True),
    ("model has at least one version", len(model_versions) >= 1, True),
])

## 4 · Where would this go next?

Not built in this chapter, and each one costs money or leaves state:

* **Score new incidents** in a job task and write predictions to a table such as `shield.gold.incident_scores`.
* **Serve the model** on a serving endpoint. Custom CPU model serving is listed for Sweden Central in the Microsoft Learn region table, but not every region has it, and an endpoint bills while it is up. Check availability before promising one.
* **Ask questions in plain English** with Genie on the gold tables (see the status note in the serving-layer section).
* **Refresh the dashboard from a job** with a dashboard task (mini-project 2 in the doc).

## 👀 In the workspace

* **Dashboards**: your published dashboard, with its three datasets under the **Data** tab.
* **Experiments**: the notebook's experiment with the run `serious_incident_rf` and its `auc` metric. The wording of the sidebar entry differs between MLflow versions.
* **Catalog** → `shield` → `gold` → **Models** → `serious_incident_model`: one version per run of Step 5; its page links back to the run.
* **SQL Warehouses**: the state of the warehouse the dashboard uses (**Running** costs money; **Stopped** does not).

## 🧹 Clean up

Chapter 16 does not need anything from this chapter. The silver and gold tables stay (chapter 16 reads them). Everything else this chapter made costs money or clutters the workspace:

| What | Created by | How to remove it |
|---|---|---|
| Registered model and all its versions | Step 5 | the cell below |
| Dashboard | Step 2 (UI) | move it to trash from the dashboard list or the workspace browser |
| SQL warehouse (if you created one just for this) | Step 2 (UI) | stop it, or delete it under **SQL Warehouses** |
| MLflow runs | Step 5 | delete the runs from the **Experiments** page (small, free, but they pile up) |

The cell deletes the model only when you set `CLEAN_UP = True`, so **Run all** never removes it by accident. Deleting a model cannot be undone.

In [ ]:
CLEAN_UP = False  # set to True to delete the registered model and all its versions

if CLEAN_UP:
    client = mlflow.MlflowClient()
    versions_to_delete = client.search_model_versions(f"name = '{MODEL_NAME}'")
    for version in versions_to_delete:
        client.delete_model_version(name=MODEL_NAME, version=version.version)
    if versions_to_delete:
        client.delete_registered_model(name=MODEL_NAME)
    print(f"Deleted {MODEL_NAME} ({len(versions_to_delete)} versions).")
else:
    print("CLEAN_UP is False: nothing deleted.")

## 🎯 Recap

* Gold has consumers now: three dataset queries for an AI/BI dashboard, checked here and built in the UI, and a model trained on the 9,500 validated incidents from silver.
* The model scores AUC ≈ 0.5 because the data is random. Reporting that honestly is what a healthy ML pipeline does.
* The model lives in Unity Catalog as `shield.gold.serious_incident_model`, one version per run. Dashboards, warehouses and models cost money, so stop and delete what you built.

Mini-projects: [docs/15-avengers-initiative.md](../../docs/15-avengers-initiative.md#try-it-yourself)

> 💡 Next: chapter 16 is the capstone: a health check that fails the job when the platform looks wrong.